<div align="center">

# BUSINESS ANALYTICS PROJECT

## Business Analytics for Supermarket Customer and Sales Analysis

**Laxmi Charitable Trust's**
**Sheth L.U.J College of Arts & Sir M.V. College of Science and Commerce**
Department of Information Technology (B.Sc.I.T Semester V)

</div>

| | |
|---|---|
| **Student Name** | Sagar Kandari |
| **Roll Number** | T010 |
| **Class** | TYIT |
| **Batch** | 2 |
| **Subject** | Applied Business Analytics |
| **Academic Year** | 2026-27 |
| **Faculty Name** | ______________________ |

## Table of Contents
1. Introduction  2. Problem Statement  3. Objectives  4. Business Scenario  5. Dataset Description  6. Data Collection
7. Data Preprocessing  8. Exploratory Data Analysis  9. Descriptive Statistics  10. Distribution Analysis
11. Correlation Analysis  12. Regression Analysis  13. BI Dashboard  14. Probability  15. Decision Making
16. Sampling  17. Confidence Intervals  18. Bootstrapping  19. Hypothesis Testing  20. A/B Testing
21. Time Series  22. Forecast Accuracy  23. Optimization  24. Simulation  25. ANOVA
26. Experimental Design  27. Factorial Experiment  28. Business Insights  29. Recommendations  30. Conclusion

## 1. Introduction
**What is Business Analytics?** Business Analytics is the use of data, statistics and software to understand business performance and to support better decisions.

**Why is data important?** Data shows what customers buy, when they buy and how much they spend. Decisions based on data are more accurate than decisions based on guesses.

**Business area selected:** Retail - a supermarket chain.

**Why this business?** A supermarket produces a large number of daily transactions, so it gives enough data for every technique in this project.

**How can analytics help?** It can show the best-selling products, the best branches, customer spending patterns, future sales and the best stock level to keep.

## 2. Problem Statement
To analyze the sales data of a supermarket to identify customer purchasing patterns, the factors that affect sales, future sales trends, and suitable business strategies.
- **Problem:** The supermarket does not know which products, branches and customers bring the most sales.
- **Data available:** 500 invoices (Jan-Mar 2026) with product, branch, price, quantity, payment type, rating and sales.
- **To analyze:** sales patterns, relationships between variables, future sales.
- **Decisions needed:** which products to promote, how much stock to keep, which payment and marketing methods to focus on.

## 3. Objectives
1. To collect and clean the supermarket sales data.
2. To perform exploratory data analysis.
3. To identify relationships among business variables.
4. To predict future sales.
5. To perform statistical testing.
6. To analyze customer behaviour.
7. To identify suitable business strategies.
8. To support business decision-making using data.

## 4. Business Scenario
- **Business name:** FreshMart Supermarket (hypothetical)
- **Type:** Retail supermarket with 3 branches - Andheri, Borivali and Thane
- **Products:** Groceries, Beverages, Snacks, Personal Care, Home Care
- **Target customers:** Families and working professionals (Member and Normal customers)
- **Operations:** Daily billing, stock purchase, payment through Cash, Card and UPI
- **Important problems:** Choosing stock levels, finding high-value products and planning promotions

## 5. Dataset Description
**Dataset name:** supermarket_sales.csv | **Source:** Sample supermarket sales data created for this project (same layout as public Kaggle supermarket datasets) | **Records:** 500 | **Columns:** 11

| Column | Description | Data Type |
|---|---|---|
| Invoice_ID | Unique invoice number | Categorical |
| Date | Date of purchase | Date/Time |
| Branch | Store branch | Categorical |
| Customer_Type | Member or Normal | Categorical |
| Gender | Customer gender | Categorical |
| Product_Line | Product category | Categorical |
| Unit_Price | Price of one unit (Rs.) | Numerical |
| Quantity | Units purchased | Numerical |
| Payment | Cash, Card or UPI | Categorical |
| Rating | Customer rating (4-10) | Numerical |
| Sales | Unit_Price x Quantity (Rs.) | Numerical |

**Numerical:** Unit_Price, Quantity, Rating, Sales  |  **Categorical:** Invoice_ID, Branch, Customer_Type, Gender, Product_Line, Payment  |  **Date/Time:** Date

## 6. Data Collection
Data is business-generated transaction data (point-of-sale billing). Collection method: sample dataset prepared for academic use. Records: 500. Time period: 1 Jan 2026 to 31 Mar 2026. All 11 variables listed above were collected.

---
## Code: Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# If the dataset file is not present, create a sample supermarket dataset
if not os.path.exists("supermarket_sales.csv"):
    np.random.seed(42)
    n = 500
    lines = ["Groceries", "Beverages", "Snacks", "Personal Care", "Home Care"]
    price_range = {"Groceries": (20, 120), "Beverages": (15, 90), "Snacks": (10, 60),
                   "Personal Care": (40, 250), "Home Care": (30, 200)}
    product = np.random.choice(lines, n)
    price = [round(np.random.uniform(*price_range[p]), 2) for p in product]
    data = pd.DataFrame({
        "Invoice_ID": ["INV" + str(1000 + i) for i in range(n)],
        "Date": np.random.choice(pd.date_range("2026-01-01", "2026-03-31").strftime("%Y-%m-%d"), n),
        "Branch": np.random.choice(["Andheri", "Borivali", "Thane"], n),
        "Customer_Type": np.random.choice(["Member", "Normal"], n, p=[0.45, 0.55]),
        "Gender": np.random.choice(["Male", "Female"], n),
        "Product_Line": product,
        "Unit_Price": price,
        "Quantity": np.random.randint(1, 11, n),
        "Payment": np.random.choice(["Cash", "Card", "UPI"], n, p=[0.25, 0.30, 0.45]),
        "Rating": np.round(np.random.uniform(4, 10, n), 1)})
    data["Sales"] = (data["Unit_Price"] * data["Quantity"]).round(2)
    data.loc[[10, 55, 120, 300, 410], "Rating"] = np.nan          # a few missing values
    data = pd.concat([data, data.iloc[[3, 77, 200]]])             # a few duplicate rows
    data.to_csv("supermarket_sales.csv", index=False)

df = pd.read_csv("supermarket_sales.csv")
print("Dataset loaded successfully")
print("Records and Columns:", df.shape)
print("Sagar Kandari T010")

## 7. Data Preprocessing

In [ ]:
# 1. Missing values
print("Missing Values:")
print(df.isnull().sum())

# 2. Duplicate records
print("\nDuplicate Records:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

# Fill missing Rating with median
df["Rating"] = df["Rating"].fillna(df["Rating"].median())

# 3. Correct data type of Date
df["Date"] = pd.to_datetime(df["Date"])

# 4. Invalid values (price or quantity zero or negative)
invalid = df[(df["Unit_Price"] <= 0) | (df["Quantity"] <= 0)]
print("Invalid Records:", len(invalid))

# 5. Outliers in Sales using IQR
Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["Sales"] < lower) | (df["Sales"] > upper)]
print("Outliers in Sales:", len(outliers))

# 6. Inconsistent text values
for col in ["Branch", "Customer_Type", "Gender", "Product_Line", "Payment"]:
    df[col] = df[col].str.strip().str.title()

print("\nAfter cleaning:")
print("Shape:", df.shape)
print(df.isnull().sum().sum(), "missing values left")
print("Sagar Kandari T010")

**Observation:** 5 missing Ratings were filled with the median, 3 duplicate rows were removed, the Date column was converted to date format, and no invalid values were found. 35 high-value invoices were identified as outliers; they are genuine bulk purchases, so they were kept.

## 8. Exploratory Data Analysis

In [ ]:
print("First Five Records:")
print(df.head())
print("\nLast Five Records:")
print(df.tail())
print("\nShape:", df.shape)
print("\nColumn Names:", list(df.columns))
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Records:", df.duplicated().sum())
print("\nUnique Values:")
print(df.nunique())
print("\nBasic Statistics:")
print(df.describe())
print("Sagar Kandari T010")

In [ ]:
# Bar chart 1: Sales by Product Line
plt.figure(figsize=(7,5))
df.groupby("Product_Line")["Sales"].sum().sort_values().plot(kind="bar", color="steelblue")
plt.title("Total Sales by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Total Sales (Rs.)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Bar chart 2: Sales by Branch
plt.figure(figsize=(7,5))
df.groupby("Branch")["Sales"].sum().plot(kind="bar", color="orange")
plt.title("Total Sales by Branch")
plt.xlabel("Branch")
plt.ylabel("Total Sales (Rs.)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Observation:** Personal Care and Home Care bring the highest total sales. Snacks bring the lowest. Andheri and Borivali have almost equal sales, while Thane is lower.

In [ ]:
# Histogram 1: Sales
plt.figure(figsize=(7,5))
plt.hist(df["Sales"], bins=20, edgecolor="black")
plt.title("Histogram of Sales")
plt.xlabel("Sales (Rs.)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Histogram 2: Rating
plt.figure(figsize=(7,5))
plt.hist(df["Rating"], bins=10, edgecolor="black", color="green")
plt.title("Histogram of Customer Rating")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Observation:** Sales is right-skewed - most bills are small and a few bills are very large. Ratings are spread fairly evenly between 4 and 10.

In [ ]:
# Box plot
plt.figure(figsize=(7,4))
plt.boxplot(df["Sales"], vert=False)
plt.title("Box Plot of Sales")
plt.xlabel("Sales (Rs.)")
plt.tight_layout()
plt.show()

# Pie chart
plt.figure(figsize=(6,6))
df["Payment"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Payment Method Share")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Scatter plot
plt.figure(figsize=(7,5))
plt.scatter(df["Quantity"], df["Sales"])
plt.title("Quantity vs Sales")
plt.xlabel("Quantity")
plt.ylabel("Sales (Rs.)")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Observation:** The box plot shows many high outliers in Sales. UPI is the most used payment method (about 45%), followed by Card and Cash. The scatter plot shows that sales rise as quantity increases.

## 9. Descriptive Statistical Analysis

In [ ]:
cols = ["Unit_Price", "Quantity", "Sales", "Rating"]

desc = pd.DataFrame({
    "Mean": df[cols].mean(),
    "Median": df[cols].median(),
    "Mode": df[cols].mode().iloc[0],
    "Minimum": df[cols].min(),
    "Maximum": df[cols].max(),
    "Range": df[cols].max() - df[cols].min(),
    "Variance": df[cols].var(),
    "Std Dev": df[cols].std()
})
print(desc.round(2))
print("Sagar Kandari T010")

**Interpretation:** Mean sales (about Rs. 455) is higher than the median (about Rs. 311), so a few large bills pull the average up. The standard deviation is large, which means bill amounts vary a lot. Average quantity is about 5 units per bill.

## 10. Distribution Analysis

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(df["Sales"], bins=20, edgecolor="black")
plt.title("Distribution of Sales")
plt.xlabel("Sales (Rs.)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.boxplot(df["Sales"], vert=False)
plt.title("Box Plot of Sales")
plt.xlabel("Sales (Rs.)")
plt.tight_layout()
plt.show()

print("Mean:", round(df["Sales"].mean(), 2))
print("Median:", round(df["Sales"].median(), 2))
print("Standard Deviation:", round(df["Sales"].std(), 2))
print("Skewness:", round(df["Sales"].skew(), 2))
print("Outliers:", len(df[(df["Sales"] < lower) | (df["Sales"] > upper)]))
print("Sagar Kandari T010")

**Interpretation:** Sales is positively skewed (skewness about 1.64). Mean > median, spread is wide and there are 35 outliers on the higher side.

## 11. Correlation Analysis

In [ ]:
corr = df[["Unit_Price", "Quantity", "Sales", "Rating"]].corr()
print("Correlation Matrix:")
print(corr.round(2))

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** Sales has a strong positive relationship with Unit_Price (0.72) and a moderate positive relationship with Quantity (0.57). Rating has almost no relationship with any variable. Unit_Price and Quantity are not related to each other (0.00). No variable has a notable negative relationship.

## 12. Regression Analysis (Quantity -> Sales)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X = df[["Quantity"]]
y = df["Sales"]

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print("Regression Equation: Sales =", round(model.intercept_, 2), "+", round(model.coef_[0], 2), "x Quantity")
print("Regression Coefficient:", round(model.coef_[0], 2))
print("R2 Score:", round(r2_score(y, y_pred), 3))
mse = mean_squared_error(y, y_pred)
print("MSE:", round(mse, 2))
print("RMSE:", round(np.sqrt(mse), 2))

# Prediction for Quantity = 8
pred = model.predict(pd.DataFrame({"Quantity": [8]}))
print("Predicted Sales for Quantity 8:", round(pred[0], 2))

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.5)
plt.plot(X, y_pred, color="red")
plt.title("Linear Regression: Quantity vs Sales")
plt.xlabel("Quantity")
plt.ylabel("Sales (Rs.)")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** For every extra unit purchased, sales increase by about Rs. 86. R2 is about 0.32, so quantity explains about 32% of the variation in sales; the rest depends on price. The model predicts about Rs. 687 for a bill of 8 units.

## 13. Business Intelligence Dashboard

In [ ]:
# Save cleaned data to import into Tableau / Power BI
df.to_csv("dashboard_data.csv", index=False)
print("dashboard_data.csv saved successfully")
print("Sagar Kandari T010")

Open **Power BI** or **Tableau** and load `dashboard_data.csv`. Create:
- **KPI cards:** Total Sales, Number of Invoices, Average Rating
- **Bar chart:** Sales by Product_Line
- **Pie chart:** Sales by Payment
- **Line chart:** Sales by Date
- **Filters (slicers):** Branch, Customer_Type, Product_Line

Add a title and data labels on each chart, and paste the dashboard screenshot below.

**Insights from dashboard (write after viewing your dashboard):**
1. Personal Care is the top product line by sales.
2. Andheri and Borivali are the strongest branches.
3. UPI is the most used payment method.
4. January had the highest sales of the three months.
5. Members spend more per bill than Normal customers.

## 14. Probability and Probability Distributions

In [ ]:
np.random.seed(1)

# Dice rolling experiment
rolls = np.random.randint(1, 7, 1000)
print("P(rolling a 6):", (rolls == 6).mean())
print("P(even number):", (rolls % 2 == 0).mean())

# Customer purchase probabilities from the dataset
print("P(customer is Member):", round((df["Customer_Type"] == "Member").mean(), 3))
print("P(Sales above 500):", round((df["Sales"] > 500).mean(), 3))
print("P(UPI payment):", round((df["Payment"] == "Upi").mean(), 3))

# Random values from three distributions
normal = np.random.normal(100, 15, 1000)
uniform = np.random.uniform(0, 100, 1000)
exponential = np.random.exponential(10, 1000)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.hist(normal, bins=20, edgecolor="black"); plt.title("Normal")
plt.subplot(1,3,2); plt.hist(uniform, bins=20, edgecolor="black"); plt.title("Uniform")
plt.subplot(1,3,3); plt.hist(exponential, bins=20, edgecolor="black"); plt.title("Exponential")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** The dice probability is close to 1/6 and the even-number probability is close to 0.5. About 46% of customers are Members and 34% of bills are above Rs. 500. Normal data is bell shaped, Uniform data is flat, and Exponential data has many small values and a long tail to the right (like waiting times).

## 15. Decision Making Under Uncertainty

```
Decision
 |-- Launch Premium Product -> High (0.3): +80 | Medium (0.5): +30 | Low (0.2): -40   EV = 31.0
 |-- Launch Budget Product  -> High (0.3): +50 | Medium (0.5): +25 | Low (0.2): -10   EV = 25.5
 |-- Do Not Launch          -> 0                                                       EV = 0
```

In [ ]:
# Payoffs in Rs. lakh with probabilities (High, Medium, Low demand)
prob = [0.3, 0.5, 0.2]

payoff = {
    "Launch Premium Product": [80, 30, -40],
    "Launch Budget Product": [50, 25, -10],
    "Do Not Launch": [0, 0, 0]
}

expected = {}
for alt in payoff:
    ev = sum(p * v for p, v in zip(prob, payoff[alt]))
    expected[alt] = ev
    print(alt, "-> Expected Value =", round(ev, 2), "lakh")

best = max(expected, key=expected.get)
print("\nBest Decision:", best)

plt.figure(figsize=(7,5))
plt.bar(expected.keys(), expected.values(), color=["green", "orange", "grey"])
plt.title("Expected Value of Each Alternative")
plt.ylabel("Expected Value (Rs. lakh)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** The Premium product has the highest expected value (Rs. 31 lakh), but it also has the largest possible loss. The Budget product is safer.

## 16. Sampling and Sampling Distribution
**Population:** all 500 invoices | **Sample:** 30 invoices | **Technique:** simple random sampling | **Repeated:** 100 times

In [ ]:
np.random.seed(42)
sales = df["Sales"]

print("Population Size:", len(sales))
print("Population Mean:", round(sales.mean(), 2))

# 100 samples of size 30
sample_means = []
for i in range(100):
    sample = sales.sample(n=30, replace=False)
    sample_means.append(sample.mean())

print("\nMean of Sample Means:", round(np.mean(sample_means), 2))
print("Standard Deviation of Sample Means:", round(np.std(sample_means, ddof=1), 2))
print("Standard Error:", round(sales.std() / np.sqrt(30), 2))

plt.figure(figsize=(7,5))
plt.hist(sample_means, bins=10, edgecolor="black")
plt.title("Sampling Distribution of Sample Means (n = 30)")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Different sample sizes
results = []
for n in [10, 20, 30, 40]:
    means = []
    for i in range(100):
        means.append(sales.sample(n=n, replace=False).mean())
    results.append({"Sample Size": n,
                    "Mean of Sample Means": np.mean(means),
                    "SD of Sample Means": np.std(means, ddof=1)})

print("\nComparison of Different Sample Sizes:")
print(pd.DataFrame(results).round(2))
print("Sagar Kandari T010")

**Central Limit Theorem:** When many random samples are taken, the sample means form a bell-shaped distribution centred on the population mean. As sample size increases, the spread (standard error) becomes smaller - as seen above (SD falls from n=10 to n=40).

## 17. Confidence Interval Estimation

In [ ]:
np.random.seed(42)
sample = df["Sales"].sample(n=100, random_state=42)

mean = sample.mean()
se = stats.sem(sample)

results = []
for level in [0.90, 0.95, 0.99]:
    low, high = stats.t.interval(level, len(sample) - 1, loc=mean, scale=se)
    results.append({"Confidence Level": str(int(level * 100)) + "%",
                    "Lower Limit": low, "Upper Limit": high})

print("Sample Mean:", round(mean, 2))
print(pd.DataFrame(results).round(2))
print("Sagar Kandari T010")

**Meaning of 95% CI:** We are 95% confident that the true average sales per invoice lies between the lower and upper limits shown. A higher confidence level gives a wider interval.

## 18. Bootstrapping

In [ ]:
np.random.seed(42)
sales = df["Sales"]

boot_medians = []
for i in range(1000):
    boot_sample = sales.sample(n=len(sales), replace=True)
    boot_medians.append(boot_sample.median())

low = np.percentile(boot_medians, 2.5)
high = np.percentile(boot_medians, 97.5)

print("Original Median:", round(sales.median(), 2))
print("Mean of Bootstrap Medians:", round(np.mean(boot_medians), 2))
print("95% Bootstrap Confidence Interval:", round(low, 2), "to", round(high, 2))

plt.figure(figsize=(7,5))
plt.hist(boot_medians, bins=20, edgecolor="black")
plt.axvline(low, color="red")
plt.axvline(high, color="red")
plt.title("Bootstrap Distribution of Median Sales")
plt.xlabel("Median Sales")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** After 1,000 resamples with replacement, the median sales per invoice is estimated at about Rs. 311, with a 95% interval of about Rs. 281 to Rs. 341.

## 19. Hypothesis Testing
**Business question:** Is the average sales per invoice different from Rs. 300?

In [ ]:
# H0: Mean sales per invoice = 300
# H1: Mean sales per invoice is not equal to 300
alpha = 0.05

t_stat, p_value = stats.ttest_1samp(df["Sales"], 300)

print("Sample Mean:", round(df["Sales"].mean(), 2))
print("Test Statistic:", round(t_stat, 3))
print("p-value:", round(p_value, 4))
print("Significance Level:", alpha)

if p_value < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")
print("Sagar Kandari T010")

**Business interpretation:** The p-value is far below 0.05, so average sales per invoice is significantly different from Rs. 300 - it is higher (about Rs. 455).

## 20. A/B Testing
**H0:** Both buttons have the same conversion rate. **H1:** The conversion rates are different.

In [ ]:
# Version A: "Buy Now"   Version B: "Get Started"
visitors_a, visitors_b = 1000, 1000
conv_a, conv_b = 120, 150

rate_a = conv_a / visitors_a
rate_b = conv_b / visitors_b
print("Conversion Rate A:", rate_a)
print("Conversion Rate B:", rate_b)

# Two proportion z-test
p_pool = (conv_a + conv_b) / (visitors_a + visitors_b)
se = np.sqrt(p_pool * (1 - p_pool) * (1/visitors_a + 1/visitors_b))
z = (rate_b - rate_a) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print("z-statistic:", round(z, 3))
print("p-value:", round(p_value, 4))

if p_value < 0.05:
    print("Decision: Reject H0 - the two versions perform differently")
else:
    print("Decision: Fail to reject H0 - no significant difference")

plt.figure(figsize=(6,5))
plt.bar(["A: Buy Now", "B: Get Started"], [rate_a * 100, rate_b * 100], color=["grey", "green"])
plt.title("A/B Test - Conversion Rate")
plt.ylabel("Conversion Rate (%)")
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Business interpretation:** Version B ("Get Started") converts at 15% versus 12% for A. The result is only just significant (p is close to 0.05), so B can be adopted but should be tested a little longer.

## 21. Time Series Analysis

In [ ]:
# Convert date, set as index and total daily sales
df["Date"] = pd.to_datetime(df["Date"])
daily = df.groupby("Date")["Sales"].sum()
daily = daily.asfreq("D").fillna(0)

plt.figure(figsize=(10,4))
plt.plot(daily)
plt.title("Daily Sales")
plt.xlabel("Date")
plt.ylabel("Sales (Rs.)")
plt.tight_layout()
plt.show()

# Trend using 7-day moving average
moving_avg = daily.rolling(7).mean()
plt.figure(figsize=(10,4))
plt.plot(daily, label="Daily Sales")
plt.plot(moving_avg, color="red", label="7-day Moving Average")
plt.title("Trend of Sales")
plt.legend()
plt.tight_layout()
plt.show()

# Seasonality: average sales by day of week
weekday = daily.groupby(daily.index.day_name()).mean()
print(weekday.round(2))
print("Sagar Kandari T010")

**Observation:** Daily sales move up and down, with the highest average sales on Sunday and Saturday and the lowest on Wednesday, suggesting a weekly (weekend) pattern. January sales were the highest month.

In [ ]:
# pip install statsmodels   (run once if not installed)
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA

# Keep last 14 days for testing
train = daily[:-14]
test = daily[-14:]

# Exponential smoothing forecast
es_model = SimpleExpSmoothing(train).fit()
es_forecast = es_model.forecast(14)

# ARIMA forecast
arima_model = ARIMA(train, order=(1, 1, 1)).fit()
arima_forecast = arima_model.forecast(14)

print("Exponential Smoothing Forecast:")
print(es_forecast.round(2).head())
print("\nARIMA Forecast:")
print(arima_forecast.round(2).head())

plt.figure(figsize=(10,4))
plt.plot(test, label="Actual")
plt.plot(es_forecast, label="Exponential Smoothing")
plt.plot(arima_forecast, label="ARIMA")
plt.title("Forecast Comparison")
plt.legend()
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Observation:** Exponential smoothing gives an almost flat forecast near the recent average. ARIMA also gives a smooth forecast. Compare both with the actual line in the chart above.

## 22. Forecast Accuracy

In [ ]:
from sklearn.metrics import mean_absolute_error

rows = []
for name, forecast in [("Exponential Smoothing", es_forecast), ("ARIMA", arima_forecast)]:
    mae = mean_absolute_error(test, forecast)
    mse = mean_squared_error(test, forecast)
    rows.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": np.sqrt(mse)})

print(pd.DataFrame(rows).round(2))
print("Sagar Kandari T010")

**Interpretation:** MAE is the average error in rupees, and RMSE gives more weight to large errors. The model with the lower MAE and RMSE is the better forecaster.

## 23. Optimization Modeling
**Decision variables:** A, B = units of Product A and B | **Objective:** Maximize Profit = 40A + 30B | **Constraints:** 2A + B <= 100 (labour), A + 2B <= 80 (machine) | **Non-negativity:** A, B >= 0

In [ ]:
from scipy.optimize import linprog

# Maximize Profit = 40A + 30B
# Labour:  2A + 1B <= 100
# Machine: 1A + 2B <= 80
# A, B >= 0
c = [-40, -30]                 # negative because linprog minimizes
A_ub = [[2, 1], [1, 2]]
b_ub = [100, 80]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None))

A, B = result.x
print("Optimal units of Product A:", round(A, 2))
print("Optimal units of Product B:", round(B, 2))
print("Maximum Profit: Rs.", round(-result.fun, 2))
print("Labour used:", round(2*A + B, 2), "out of 100")
print("Machine used:", round(A + 2*B, 2), "out of 80")
print("Sagar Kandari T010")

**Business meaning:** Produce 40 units of A and 20 units of B for a maximum profit of Rs. 2,200. Both labour and machine hours are fully used, so adding either resource would increase profit.

## 24. Simulation Modeling
Inventory simulation for 365 days with random daily demand (average 40 units). Three scenarios: stock of 35, 45 and 55 units per day.

In [ ]:
np.random.seed(5)

def simulate(stock, days=365):
    demand = np.random.poisson(40, days)          # daily demand
    sold = np.minimum(demand, stock)
    shortage = np.maximum(demand - stock, 0)
    leftover = stock - sold
    profit = sold * 20 - shortage * 5 - leftover * 2
    return {"Total Sales (units)": sold.sum(),
            "Shortage (units)": shortage.sum(),
            "Average Inventory": leftover.mean(),
            "Service Level %": 100 * sold.sum() / demand.sum(),
            "Profit (Rs.)": profit.sum()}

scenarios = {"Stock 35": 35, "Stock 45": 45, "Stock 55": 55}
result = pd.DataFrame({name: simulate(s) for name, s in scenarios.items()}).T
print(result.round(1))

result["Profit (Rs.)"].plot(kind="bar", color="teal")
plt.title("Profit for Different Stock Levels")
plt.ylabel("Profit (Rs.)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** Stocking 35 units causes many shortages (service level about 86%). Stocking 45 units gives the highest profit with about 98% service level. Stocking 55 units adds very little service but extra leftover stock, so profit drops.

## 25. ANOVA
**H0:** Mean sales of Cash, Card and UPI bills are equal. **H1:** At least one mean is different.

In [ ]:
# H0: Mean sales of Cash, Card and UPI payments are equal
# H1: At least one mean is different
cash = df[df["Payment"] == "Cash"]["Sales"]
card = df[df["Payment"] == "Card"]["Sales"]
upi = df[df["Payment"] == "Upi"]["Sales"]

f_stat, p_value = stats.f_oneway(cash, card, upi)

print("Mean Sales - Cash:", round(cash.mean(), 2), "Card:", round(card.mean(), 2), "UPI:", round(upi.mean(), 2))
print("F-statistic:", round(f_stat, 3))
print("p-value:", round(p_value, 4))
print("Significance Level: 0.05")

if p_value < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")
print("Sagar Kandari T010")

**Business interpretation:** The p-value is above 0.05, so there is no significant difference in spending between payment methods. Promotions need not be different for each payment type.

## 26. Experimental Design
- **Factor 1:** Advertising Type - Social Media, Email
- **Factor 2:** Discount - 5%, 15%, 25%
- **Response variable:** Average sales per store
- **Experimental units:** 60 store-weeks (10 for each combination)
- **Conduct:** Each store-week is randomly assigned one ad type and one discount level; sales are recorded at the end of the week.

In [ ]:
# Experiment table: every combination of the two factors
factor1 = ["Social Media", "Email"]
factor2 = ["5% Discount", "15% Discount", "25% Discount"]

design = pd.DataFrame([(a, d) for a in factor1 for d in factor2],
                      columns=["Advertising Type", "Discount"])
design["Stores per combination"] = 10
print(design)
print("Total experimental units:", design["Stores per combination"].sum())
print("Sagar Kandari T010")

## 27. Factorial Experiment (2 x 3)

In [ ]:
# pip install statsmodels   (run once if not installed)
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(7)
rows = []
for ad, ad_effect in [("Social", 5), ("Email", 0)]:
    for disc, disc_effect in [(5, 0), (15, 10), (25, 15)]:
        for i in range(10):
            sales_value = 100 + ad_effect + disc_effect + np.random.normal(0, 5)
            rows.append([ad, disc, sales_value])

exp = pd.DataFrame(rows, columns=["Ad", "Discount", "Sales"])

model = ols("Sales ~ C(Ad) + C(Discount) + C(Ad):C(Discount)", data=exp).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table.round(4))

# Interaction plot
means = exp.groupby(["Discount", "Ad"])["Sales"].mean().unstack()
means.plot(marker="o", figsize=(7,5))
plt.title("Interaction Plot: Advertising Type x Discount")
plt.xlabel("Discount (%)")
plt.ylabel("Average Sales")
plt.xticks([5, 15, 25])
plt.tight_layout()
plt.show()
print("Sagar Kandari T010")

**Interpretation:** Check the p-values (PR(>F)) in the table. A p-value below 0.05 for Ad means advertising type affects sales, for Discount means discount level affects sales, and for Ad:Discount means the effect of one factor depends on the other. In the interaction plot, lines that are not parallel show an interaction.

## 28. Business Insights and Findings
1. **Personal Care** and **Home Care** give the highest sales (Rs. 88,078 and Rs. 61,335), because their average bill (Rs. 786 and Rs. 660) is much higher than Snacks (Rs. 164).
2. **Thane** branch sales (Rs. 66,152) are about 18% lower than Andheri (Rs. 81,198) and Borivali (Rs. 80,360).
3. **Member** customers spend more per bill (Rs. 480) than Normal customers (Rs. 434).
4. **UPI** is the most used payment method (45.4%), ahead of Card (31.6%) and Cash (23.0%).
5. Sales is right-skewed (skewness 1.64); mean Rs. 455 vs median Rs. 311, so a few large bills drive revenue.
6. **Unit price** has a strong relationship with sales (r = 0.72) and **quantity** a moderate one (r = 0.57). Regression shows each extra unit adds about Rs. 86.
7. Average sales per invoice is significantly different from Rs. 300 (t-test, p < 0.05).
8. Payment method does **not** significantly change spending (ANOVA p = 0.65).
9. Inventory simulation shows a stock level of 45 units/day gives the highest profit with about 98% service level.
10. Button B ("Get Started") converts better (15% vs 12%), at the edge of significance (p = 0.0496).

## 29. Business Recommendations
1. Promote Personal Care and Home Care products, since they give the highest sales per bill.
2. Run local promotions in the Thane branch to bring it up to the level of the other branches.
3. Encourage more customers to become Members, as Members spend more per bill.
4. Keep UPI as the main payment option and offer UPI cashback offers.
5. Keep around 45 units/day stock for fast-moving items to avoid shortages without overstocking.
6. Use sales forecasts for weekly stock planning, with extra stock for weekends.
7. Use the "Get Started" button on the website, and keep testing it.

## 30. Conclusion
This project studied the sales data of a supermarket. The data was cleaned and explored, and descriptive statistics, correlation, regression, probability, sampling, confidence intervals, bootstrapping, hypothesis testing, A/B testing, time series forecasting, optimization, simulation, ANOVA and factorial experiments were applied. The main findings are that Personal Care and Home Care drive sales, Thane lags behind, Members spend more, and a stock level of about 45 units gives the best profit. The supermarket can use these findings for stock planning, promotions and forecasting. Through this project I learned how different analytics techniques work together to support business decisions.

*Sagar Kandari - T010 - TYIT*